# A02-01 — Shortest Path Between Rooms
## Box House — Wayfinding Analysis

**Project:** Box House — Graph-ML Assignment 02  
**Author:** Symon Kipkemei  
**Date:** 2026-05-19

---

A building is only useful if its occupants can move through it. This notebook asks a direct question: given any two rooms in the Box House, what is the minimum sequence of spaces an occupant must pass through to get from one to the other — using only the actual doors and windows in the building?

Floor plan adjacency does not answer this. Two rooms can share a wall with no door between them. The circulation graph constrains every path to registered apertures only, so the route it returns is physically traversable, not hypothetical.

## 1. Import Libraries

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.Color import Color
import time

e:\softwares-4\graph-ml\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [2]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This notebook requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.21) is OLDER than the latest version (0.9.33) from PyPI. Please consider upgrading to the latest version.


## 3. Set Renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [3]:
renderer = "vscode"

## 4. Load Geometry

In [4]:
objects = Topology.ByOBJPath(
    r"E:\softwares-4\graph-ml\assign-01-graphs\geometry\box-house-rooms.obj",
    selfMerge=True
)
print("Room objects:", objects)

Room objects: [<topologic_core.Cluster object at 0x0000022344DFE370>]


In [5]:
doors   = Topology.ByOBJPath(
    r"E:\softwares-4\graph-ml\assign-01-graphs\geometry\box-house-doors.obj",
    selfMerge=True
)
windows = Topology.ByOBJPath(
    r"E:\softwares-4\graph-ml\assign-01-graphs\geometry\box-house-windows.obj",
    selfMerge=True
)

aperture_faces = []
for ap in doors:
    for f in (Topology.Faces(ap) or [ap]):
        Topology.SetDictionary(f, Dictionary.ByKeysValues(["color", "type"], ["brown", "door"]))
        aperture_faces.append(f)
for ap in windows:
    for f in (Topology.Faces(ap) or [ap]):
        Topology.SetDictionary(f, Dictionary.ByKeysValues(["color", "type"], ["cyan", "window"]))
        aperture_faces.append(f)

print("Total apertures:", len(aperture_faces))

Total apertures: 36


## 5. Build CellComplex and Add Apertures

In [6]:
cells = Topology.Cells(objects[0])
cc = CellComplex.ByCells(cells)
cc = Topology.RemoveCoplanarFaces(cc)
cc = Topology.RemoveCollinearEdges(cc)
cc = Topology.AddApertures(cc, aperture_faces, subTopologyType="face")
print("Cells:", len(cells))
print("Apertures registered:", len(aperture_faces))

Cells: 19
Apertures registered: 36


## 6. Build Circulation Graph

The circulation graph models the movement network of the building. A room is a node. An edge exists between two rooms only if a registered door or window sits on their shared wall. There are no speculative connections — only openings that were explicitly modelled.

Exterior-facing apertures are excluded here. The focus is on interior circulation: how occupants move between rooms, not how they enter or exit the building.

In [7]:
g_circ = Graph.ByTopology(
    cc,
    direct=False,
    viaSharedApertures=True,
    toExteriorApertures=False
)
print("Vertices:", len(Graph.Vertices(g_circ)))
print("Edges:   ", len(Graph.Edges(g_circ)))

Vertices: 39
Edges:    40


## 7. Circulation Network

39 nodes, 40 edges. The graph shows the complete interior movement network of the building — every room and every aperture-based connection between them.

In [8]:
for v in Graph.Vertices(g_circ):
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size", "color"], [16, "red"]))
for e in Graph.Edges(g_circ):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [3, "white"]))

ap_cluster = Cluster.ByTopologies(aperture_faces)
Topology.Show(
    cc, ap_cluster, g_circ,
    faceColorKey="color",
    vertexSizeKey="size",
    vertexColorKey="color",
    edgeWidthKey="width",
    edgeColorKey="color",
    faceOpacity=0.15,
    backgroundColor="black",
    width=800,
    height=600,
    renderer=renderer
)

## 8. Room Index Reference

The Box House OBJ has no named room groups. Each room is identified by its graph vertex index and 3D centroid position. This table is the reference for all subsequent analysis — use it to map index numbers back to physical locations in the building model.

In [9]:
g_verts = Graph.Vertices(g_circ)
print(f"{'Index':>6}  {'X':>8}  {'Y':>8}  {'Z':>8}")
print("-" * 38)
for i, v in enumerate(g_verts):
    print(f"{i:>6}  {v.X():>8.2f}  {v.Y():>8.2f}  {v.Z():>8.2f}")

 Index         X         Y         Z
--------------------------------------
     0  16310.01  -4669.99   4170.17
     1   7763.92   2240.01   1022.61
     2   8992.09   -759.99   4022.61
     3   7310.01   1330.01   1170.17
     4   7310.01  -2211.11   1170.17
     5   7310.01  -2883.85   4170.17
     6   7763.92   -759.99   1022.61
     7   7310.01    174.37   4170.17
     8  18122.72  -3759.99   4022.61
     9  16310.01  -4813.42   1170.17
    10   6562.24   -759.99   4022.61
    11  17643.50  -3759.99   1022.61
    12  19310.01  -2727.51   1170.17
    13   8810.00    740.00   4500.00
    14  17810.00  -5260.00   1500.00
    15   6410.99    740.00   4500.00
    16   8810.00  -2260.00   1500.00
    17  15202.88  -3759.99   4022.61
    18  13310.00  -2260.00   4500.00
    19  10310.01  -2084.33   4170.17
    20  20810.00  -4961.99   1500.00
    21  14810.00  -5260.00   4500.00
    22  12517.60   -759.99   4022.61
    23  17810.00  -2860.99   1500.00
    24  17810.00  -5260.00   4500.00

## 9. The Wayfinding Question

Can an occupant reach room 38 from room 0? If so, through how many intermediate spaces?

Room 0 sits at one end of the building (x≈16310, z≈4170). Room 38 is at the other end (x≈10310, z≈1170). They are on different floor levels. The path, if it exists, must thread through the entire aperture network between them.

In [10]:
START_INDEX = 0
END_INDEX   = len(g_verts) - 1

start_v = g_verts[START_INDEX]
end_v   = g_verts[END_INDEX]

print(f"Start room {START_INDEX}: ({start_v.X():.2f}, {start_v.Y():.2f}, {start_v.Z():.2f})")
print(f"End   room {END_INDEX}:   ({end_v.X():.2f}, {end_v.Y():.2f}, {end_v.Z():.2f})")

Start room 0: (16310.01, -4669.99, 4170.17)
End   room 38:   (10310.01, -2211.11, 1170.17)


In [11]:
t0  = time.time()
crg = Graph.CompiledRoutingGraph(g_circ, precomputeTurns=False)
shortest_path = Graph.ShortestPath(crg, vertexA=start_v, vertexB=end_v)
t1  = time.time()

if shortest_path:
    hops   = len(Topology.Edges(shortest_path))
    length = Wire.Length(shortest_path)
    print(f"Path found in {t1-t0:.3f}s")
    print(f"Hops (room transitions): {hops}")
    print(f"Euclidean path length:   {length:.2f} units")
else:
    print("No path found — the two rooms are not connected through apertures.")

Path found in 0.022s
Hops (room transitions): 8
Euclidean path length:   15955.01 units


## 10. Path Visualisation

The path from room 0 to room 38 passes through **8 intermediate rooms** and covers a centroid-to-centroid distance of approximately **15,955 units**. Start room — green. End room — blue. Path — yellow.

8 hops for a cross-building journey indicates the building has significant network depth. An occupant cannot take a shortcut — they must thread through the full chain of connecting spaces.

In [12]:
for v in g_verts:
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size", "color"], [14, "red"]))
Topology.SetDictionary(start_v, Dictionary.ByKeysValues(["size", "color"], [20, "lime"]))
Topology.SetDictionary(end_v,   Dictionary.ByKeysValues(["size", "color"], [20, "dodgerblue"]))

for e in Graph.Edges(g_circ):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [2, "grey"]))
if shortest_path:
    for e in Topology.Edges(shortest_path):
        Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [6, "yellow"]))

show_list = [cc, ap_cluster, g_circ]
if shortest_path:
    show_list.append(shortest_path)

Topology.Show(
    *show_list,
    faceColorKey="color",
    vertexSizeKey="size",
    vertexColorKey="color",
    edgeWidthKey="width",
    edgeColorKey="color",
    faceOpacity=0.12,
    backgroundColor="black",
    width=800,
    height=600,
    renderer=renderer
)

## 11. Travel Depth Across the Building

Querying five room pairs reveals the range of travel distances. All five pairs are connected — no room is an island in the horizontal circulation network. The shortest tested path (0→19) takes 4 hops. The longest (0→38, 2→36) take 8.

This variation confirms that the building has unequal connectivity. Some rooms are deep within the layout relative to others. Distance in the network differs substantially from distance on the floor plan.

In [13]:
n = len(g_verts)
pairs = [
    (0, n - 1),
    (0, n // 2),
    (1, n - 2),
    (2, n - 3),
    (n // 4, 3 * n // 4),
]

print(f"{'Pair':>12}  {'Hops':>6}  {'Length':>10}  Status")
print("-" * 48)
for (a_idx, b_idx) in pairs:
    if a_idx >= n or b_idx >= n or a_idx == b_idx:
        continue
    path = Graph.ShortestPath(crg, vertexA=g_verts[a_idx], vertexB=g_verts[b_idx])
    if path:
        hops   = len(Topology.Edges(path))
        length = Wire.Length(path)
        print(f"  {a_idx:>3} → {b_idx:<3}   {hops:>4}   {length:>10.2f}  connected")
    else:
        print(f"  {a_idx:>3} → {b_idx:<3}      —           —  no path")

        Pair    Hops      Length  Status
------------------------------------------------
    0 → 38       8     15955.01  connected
    0 → 19       4      8752.77  connected
    1 → 37       6     12115.19  connected
    2 → 36       8     16448.84  connected
    9 → 29       7     13400.70  connected
